# ADW — Perbandingan Linear Regression dan SARIMA pada Dua Dataset Sintetis

Notebook ini membandingkan **Linear Regression** dan **SARIMA** pada dua seri bulanan:

- **V1 — Synthetic Independent:** baseline sintetis independen yang sudah dibuat sebelumnya; 180 angka dipertahankan persis.
- **V2 — Synthetic Procurement-Inspired:** data sintetis baru yang dibangkitkan independen dengan asumsi ilustratif konteks procurement (tren, pola musiman, autokorelasi, dan perubahan skenario). V2 **tidak** menggunakan, menyalin, mengkalibrasi, atau merekonstruksi data internship/perusahaan.

Kedua dataset memakai rentang bulanan dan jumlah observasi yang sama agar perbandingan metode lebih mudah dibaca. Nilai V2 sepenuhnya hasil generator dengan seed tetap, sehingga dapat direproduksi.

## 1. Persiapan library

Jalankan sel ini terlebih dahulu. Di JupyterLite, `piplite` akan mencoba memasang `statsmodels` jika belum tersedia. `statsmodels` dipakai untuk SARIMA serta uji ADF/KPSS.

In [ ]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")

try:
    import statsmodels
except ImportError:
    try:
        import piplite
        await piplite.install("statsmodels")
    except Exception as exc:
        print("Instalasi statsmodels otomatis tidak berhasil:", repr(exc))

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.stattools import adfuller, kpss
    HAS_STATSMODELS = True
    print("statsmodels tersedia:", statsmodels.__version__ if 'statsmodels' in sys.modules else "installed")
except Exception as exc:
    HAS_STATSMODELS = False
    print("statsmodels belum tersedia. SARIMA dan ADF/KPSS belum bisa dijalankan.")
    print("Detail:", repr(exc))

## 2. Membuat / memuat V1 dan membangkitkan V2

V1 memakai angka baseline yang sudah ada, bukan dibuat ulang dengan rumus berbeda. V2 dibangkitkan dari parameter hipotetis yang tertulis di fungsi berikut. Parameter musiman hanya skenario demonstrasi, bukan klaim pola aktual procurement perusahaan.

In [ ]:
# V1: baseline lama, 180 nilai dipertahankan apa adanya.
V1_COUNTS = [
    350,358,370,366,356,331,323,320,319,313,322,333,
    342,358,368,393,377,370,360,352,343,354,356,368,
    383,377,393,384,346,352,325,320,285,301,310,330,
    347,359,366,351,341,341,326,314,309,332,333,353,
    375,373,365,363,361,334,326,319,301,319,326,324,
    336,338,347,345,354,344,349,330,316,324,316,348,
    369,371,390,386,391,386,371,363,346,353,382,394,
    418,430,437,431,433,422,417,397,386,396,398,399,
    418,439,444,439,441,406,385,388,380,374,411,411,
    434,444,466,462,465,461,446,432,428,406,422,438,
    415,418,420,430,431,407,388,369,367,374,391,413,
    432,432,442,426,431,417,405,369,357,375,375,384,
    397,416,411,423,406,393,379,372,361,367,388,422,
    443,460,465,448,439,417,415,398,395,389,384,423,
    431,443,459,457,466,446,437,420,410,411,423,439,
]

N_PERIODS = 180
START_MONTH = "2009-01-01"
SEED_V2 = 20261110

dates = pd.date_range(START_MONTH, periods=N_PERIODS, freq="MS")
v1 = pd.DataFrame({
    "PeriodMonth": dates,
    "SyntheticEventCount": np.array(V1_COUNTS, dtype=int),
    "DataStatus": "SYNTHETIC_INDEPENDENT_V1",
    "GeneratorVersion": "V1_INDEPENDENT_RANDOM_WALK",
})

def generate_v2(seed=SEED_V2, n_periods=N_PERIODS, start=START_MONTH):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=n_periods, freq="MS")
    t = np.arange(n_periods, dtype=float)
    month = dates.month.to_numpy()

    # Indeks musiman hipotetis: aktivitas relatif lebih tinggi di beberapa
    # bulan awal/akhir tahun. Ini asumsi ilustratif, bukan pola terukur perusahaan.
    seasonal_index = np.array([
        0.98, 0.96, 1.02, 1.07, 1.04, 0.96,
        0.92, 0.91, 0.95, 1.01, 1.08, 1.10
    ])
    seasonal = seasonal_index[month - 1]
    trend = 315.0 + 0.42 * t
    scenario_shift = np.where(t >= 108, 22.0, 0.0)

    # Gangguan AR(1) agar observasi berurutan saling berkaitan.
    rho = 0.58
    innovations = rng.normal(0.0, 10.5, size=n_periods)
    ar = np.zeros(n_periods, dtype=float)
    for i in range(1, n_periods):
        ar[i] = rho * ar[i - 1] + innovations[i]
    observation_noise = rng.normal(0.0, 5.5, size=n_periods)

    values = trend * seasonal + scenario_shift + ar + observation_noise
    counts = np.maximum(1, np.rint(values)).astype(int)
    return pd.DataFrame({
        "PeriodMonth": dates,
        "SyntheticEventCount": counts,
        "DataStatus": "SYNTHETIC_PROCUREMENT_INSPIRED_V2",
        "GeneratorVersion": f"V2_INDEPENDENT_SCENARIO_SEED_{seed}",
    })

v2 = generate_v2()

print("V1:", len(v1), "observasi |", v1.PeriodMonth.min().date(), "s.d.", v1.PeriodMonth.max().date())
print("V2:", len(v2), "observasi |", v2.PeriodMonth.min().date(), "s.d.", v2.PeriodMonth.max().date())
print("V1 sama dengan baseline lama: 180 angka tertanam dan tidak diubah.")
print("V2 seed:", SEED_V2)
display(pd.concat([v1.head(3), v2.head(3)], keys=["V1", "V2"]))

## 3. Visualisasi dan statistik deskriptif

In [ ]:
datasets = {
    "V1 — Independent": v1,
    "V2 — Procurement-inspired": v2,
}

fig, ax = plt.subplots(figsize=(12, 4.5))
for name, df in datasets.items():
    ax.plot(df["PeriodMonth"], df["SyntheticEventCount"], label=name, linewidth=1.5)
ax.set_title("Perbandingan Dua Dataset Sintetis")
ax.set_xlabel("Bulan")
ax.set_ylabel("Jumlah aktivitas sintetis")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

def descriptive_summary(df, name):
    x = df["SyntheticEventCount"].astype(float).to_numpy()
    return {
        "Dataset": name,
        "N": len(x),
        "Mean": x.mean(),
        "Std Dev": x.std(ddof=1),
        "Minimum": x.min(),
        "Maximum": x.max(),
        "Trend linear / bulan": np.polyfit(np.arange(len(x)), x, 1)[0],
        "ACF lag-1": np.corrcoef(x[:-1], x[1:])[0, 1],
    }

summary = pd.DataFrame([descriptive_summary(df, name) for name, df in datasets.items()])
display(summary.round(3))

## 4. Uji stasioneritas ADF dan KPSS

ADF: H0 = terdapat unit root (indikasi nonstasioner). KPSS: H0 = stasioner pada level. Gunakan keduanya sebagai informasi pendukung; keputusan pemodelan SARIMA tidak hanya ditentukan oleh satu p-value.

In [ ]:
stationarity_rows = []
if HAS_STATSMODELS:
    for name, df in datasets.items():
        x = df["SyntheticEventCount"].astype(float)
        for label, series in [("Level", x), ("First difference", x.diff().dropna())]:
            try:
                adf_p = adfuller(series, autolag="AIC")[1]
            except Exception:
                adf_p = np.nan
            try:
                kpss_p = kpss(series, regression="c", nlags="auto")[1]
            except Exception:
                kpss_p = np.nan
            stationarity_rows.append({
                "Dataset": name,
                "Transformasi": label,
                "ADF p-value": adf_p,
                "KPSS p-value": kpss_p,
            })
    stationarity = pd.DataFrame(stationarity_rows)
    display(stationarity.round(4))
else:
    print("Lewati ADF/KPSS karena statsmodels belum tersedia.")

## 5. Pembagian train-test dan fungsi evaluasi

Pemisahan dilakukan kronologis (80% data awal untuk train, 20% data terakhir untuk test). Tidak ada pengacakan baris, supaya urutan waktu tetap terjaga. Semua model menggunakan pembagian yang sama.

In [ ]:
TRAIN_RATIO = 0.80

def split_series(df):
    y = df["SyntheticEventCount"].astype(float).to_numpy()
    split_at = int(len(y) * TRAIN_RATIO)
    return y[:split_at], y[split_at:], split_at

def score_forecast(actual, predicted, dataset_name, model_name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = actual - predicted
    nonzero = np.abs(actual) > 1e-12
    mape = np.mean(np.abs(error[nonzero] / actual[nonzero])) * 100 if nonzero.any() else np.nan
    return {
        "Dataset": dataset_name,
        "Model": model_name,
        "MAE": np.mean(np.abs(error)),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "MAPE (%)": mape,
    }

## 6. Model 1 — Linear Regression

Model regresi menggunakan tren waktu dan dummy bulan (musiman bulanan). Ini adalah baseline regresi yang dapat menangkap tren linier serta perbedaan rata-rata antarbulan.

In [ ]:
def linear_regression_fit_forecast(y_train, horizon, start_index=0):
    y_train = np.asarray(y_train, dtype=float)
    n = len(y_train)
    idx = np.arange(start_index, start_index + n)
    months = idx % 12
    X = np.column_stack([
        np.ones(n), idx,
        *[(months == m).astype(float) for m in range(1, 12)]
    ])
    coef, *_ = np.linalg.lstsq(X, y_train, rcond=None)
    future_idx = np.arange(start_index + n, start_index + n + horizon)
    future_months = future_idx % 12
    X_future = np.column_stack([
        np.ones(horizon), future_idx,
        *[(future_months == m).astype(float) for m in range(1, 12)]
    ])
    return X_future @ coef

lr_predictions = {}
lr_future = {}
for name, df in datasets.items():
    y_train, y_test, split_at = split_series(df)
    lr_predictions[name] = linear_regression_fit_forecast(y_train, len(y_test), start_index=0)
    lr_future[name] = linear_regression_fit_forecast(
        df["SyntheticEventCount"].astype(float).to_numpy(), 12, start_index=0
    )

print("Linear Regression selesai untuk kedua dataset.")

## 7. Model 2 — SARIMA

SARIMA dipilih berdasarkan AIC terendah di antara beberapa kandidat yang ditentukan di awal. Pemilihan AIC hanya menggunakan data train; data test tetap disimpan untuk evaluasi akhir.

In [ ]:
SARIMA_CANDIDATES = [
    ((1, 0, 0), (1, 1, 0, 12)),
    ((0, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 1), (0, 1, 1, 12)),
    ((1, 0, 1), (0, 1, 1, 12)),
]

def select_and_forecast_sarima(y_train, horizon):
    if not HAS_STATSMODELS:
        raise ImportError("statsmodels diperlukan untuk SARIMA. Jalankan ulang sel instalasi library.")
    best_fit = None
    best_spec = None
    best_aic = np.inf
    tried = []
    for order, seasonal_order in SARIMA_CANDIDATES:
        try:
            model = SARIMAX(
                y_train,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            fit = model.fit(disp=False, maxiter=100)
            tried.append({"order": order, "seasonal_order": seasonal_order, "AIC": fit.aic})
            if np.isfinite(fit.aic) and fit.aic < best_aic:
                best_aic = fit.aic
                best_fit = fit
                best_spec = (order, seasonal_order)
        except Exception as exc:
            tried.append({"order": order, "seasonal_order": seasonal_order, "AIC": np.nan, "Error": str(exc)[:160]})
    if best_fit is None:
        raise RuntimeError("Semua kandidat SARIMA gagal. Periksa instalasi statsmodels atau kandidat model.")
    forecast = np.asarray(best_fit.get_forecast(steps=horizon).predicted_mean, dtype=float)
    return forecast, best_spec, best_aic, pd.DataFrame(tried)

sarima_predictions = {}
sarima_future = {}
sarima_details = []
sarima_search_tables = {}

if HAS_STATSMODELS:
    for name, df in datasets.items():
        y_train, y_test, split_at = split_series(df)
        pred, spec, aic, tried = select_and_forecast_sarima(y_train, len(y_test))
        sarima_predictions[name] = pred
        sarima_search_tables[name] = tried
        sarima_details.append({
            "Dataset": name,
            "Order": str(spec[0]),
            "Seasonal order": str(spec[1]),
            "Train AIC": aic,
        })
        # Refit on the full series for a 12-month illustrative future forecast.
        full_model = SARIMAX(
            df["SyntheticEventCount"].astype(float).to_numpy(),
            order=spec[0], seasonal_order=spec[1],
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False, maxiter=100)
        sarima_future[name] = np.asarray(full_model.get_forecast(steps=12).predicted_mean, dtype=float)
    display(pd.DataFrame(sarima_details).round(3))
else:
    print("SARIMA belum dijalankan karena statsmodels tidak tersedia.")

## 8. Perbandingan performa pada data test

MAE, RMSE, dan MAPE dihitung pada observasi test yang sama. Secara umum, nilai metrik yang lebih kecil menunjukkan error yang lebih rendah pada periode test.

In [ ]:
results = []
for name, df in datasets.items():
    y_train, y_test, split_at = split_series(df)
    results.append(score_forecast(y_test, lr_predictions[name], name, "Linear Regression"))
    if name in sarima_predictions:
        results.append(score_forecast(y_test, sarima_predictions[name], name, "SARIMA"))

comparison = pd.DataFrame(results).sort_values(["Dataset", "RMSE"]).reset_index(drop=True)
display(comparison.round(3))

## 9. Visualisasi hasil prediksi pada periode test

In [ ]:
for name, df in datasets.items():
    y_train, y_test, split_at = split_series(df)
    test_dates = df["PeriodMonth"].iloc[split_at:]
    plt.figure(figsize=(12, 4))
    plt.plot(test_dates, y_test, label="Aktual (test)", linewidth=2)
    plt.plot(test_dates, lr_predictions[name], label="Linear Regression", linestyle="--")
    if name in sarima_predictions:
        plt.plot(test_dates, sarima_predictions[name], label="SARIMA", linestyle="-.")
    plt.title(f"Prediksi pada Data Test — {name}")
    plt.xlabel("Bulan")
    plt.ylabel("Jumlah aktivitas sintetis")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 10. Forecast 12 bulan ke depan

Forecast berikut merupakan demonstrasi keluaran model setelah masing-masing model dilatih ulang menggunakan seluruh seri. Nilai ini tetap merupakan prediksi pada data sintetis, bukan estimasi aktivitas perusahaan.

In [ ]:
future_rows = []
for name, df in datasets.items():
    future_dates = pd.date_range(df["PeriodMonth"].iloc[-1] + pd.offsets.MonthBegin(1), periods=12, freq="MS")
    for i, dt in enumerate(future_dates):
        future_rows.append({"Dataset": name, "PeriodMonth": dt, "Model": "Linear Regression", "Forecast": lr_future[name][i]})
        if name in sarima_future:
            future_rows.append({"Dataset": name, "PeriodMonth": dt, "Model": "SARIMA", "Forecast": sarima_future[name][i]})

future_forecasts = pd.DataFrame(future_rows)
display(future_forecasts.head(12).round(2))

## 11. Simpan dataset dan hasil analisis

File CSV akan muncul di file browser JupyterLite pada direktori kerja notebook.

In [ ]:
# Export ulang dataset agar file yang diunduh berasal dari eksekusi notebook ini.
v1.to_csv("synthetic_v1_independent.csv", index=False, date_format="%Y-%m-%d")
v2.to_csv("synthetic_v2_procurement_inspired.csv", index=False, date_format="%Y-%m-%d")
summary.to_csv("adw_descriptive_summary_v1_v2.csv", index=False)
comparison.to_csv("adw_model_comparison_lr_sarima_v1_v2.csv", index=False)
future_forecasts.to_csv("adw_forecast_12_months_v1_v2.csv", index=False, date_format="%Y-%m-%d")
if HAS_STATSMODELS:
    stationarity.to_csv("adw_stationarity_adf_kpss_v1_v2.csv", index=False)
print("Selesai. Dataset dan tabel hasil tersimpan di direktori kerja notebook.")

## 12. Panduan kesimpulan

Saat menulis pembahasan, jawab pertanyaan berikut berdasarkan tabel dan grafik hasil eksekusi:

1. Apakah V1 dan V2 memiliki karakteristik tren, variasi, musiman, dan autokorelasi yang berbeda?
2. Pada masing-masing dataset, metode mana yang menghasilkan MAE/RMSE/MAPE terkecil pada data test?
3. Apakah metode terbaik sama pada V1 dan V2? Jelaskan bahwa performa bergantung pada pola data.
4. Apa keterbatasan Linear Regression tren + dummy bulan dan SARIMA pada seri sintetis ini?

**Batas interpretasi:** V2 hanya menggambarkan skenario sintetis procurement yang dirancang secara independen. Kemiripan konteks tidak membuktikan bahwa pola tersebut benar-benar terjadi pada data internship. Seluruh kesimpulan empiris dari notebook ini hanya berlaku untuk dua dataset sintetis yang dianalisis.